## 11월 데이터용 버전
- 이 노트북은 2019-Nov.csv 기준입니다.
- 테이블(데이터프레임)명에 `_11` 접미사를 붙인 사본입니다.



In [ ]:
!pip install geopandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import math
import platform
import geopandas as gpd

pd.set_option('display.float_format',"{:.2f}".format)

# OS에 따라 다른 폰트 지정
if platform.system() == 'Darwin':   # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':  # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:  # Linux (예: Colab, Ubuntu)
    plt.rcParams['font.family'] = 'NanumGothic'

df0_11 = pd.read_csv("2019-Nov.csv")
print(df0_11.shape)
print(df0_11.head())
print(df0_11.columns.tolist())


CSV 파일 존재 여부: True
CSV 파일 맞는지: True
CSV 경로: D:\_project\0409\데이터예시1\2019-Oct.csv\2019-Nov.csv
Parquet 저장 경로: D:\_project\0409\데이터예시1\2019-Oct.csv\2019-Nov.parquet
기존 Parquet 파일 사용
(67501979, 9)
                event_time event_type  product_id          category_id  \
0  2019-11-01 00:00:00 UTC       view     1003461  2053013555631882655   
1  2019-11-01 00:00:00 UTC       view     5000088  2053013566100866035   
2  2019-11-01 00:00:01 UTC       view    17302664  2053013553853497655   
3  2019-11-01 00:00:01 UTC       view     3601530  2053013563810775923   
4  2019-11-01 00:00:01 UTC       view     1004775  2053013555631882655   

               category_code   brand  price    user_id  \
0     electronics.smartphone  xiaomi 489.07  520088904   
1  appliances.sewing_machine  janome 293.65  530496790   
2                       None   creed  28.31  561587266   
3  appliances.kitchen.washer      lg 712.87  518085591   
4     electronics.smartphone  xiaomi 183.27  558856683   

            

In [4]:
df0_11.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     object 
 1   event_type     object 
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  object 
 5   brand          object 
 6   price          float64
 7   user_id        int64  
 8   user_session   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 4.5+ GB


In [5]:
smartphone_rows_11 = (df0_11['category_code'] == 'electronics.smartphone').sum()

ratio = smartphone_rows_11 / len(df0_11) * 100

print("전체 행 수:", len(df0_11))
print("스마트폰 행 수:", smartphone_rows_11)
print(f"전체 대비 스마트폰 비중: {ratio:.2f}%")

전체 행 수: 67501979
스마트폰 행 수: 16375000
전체 대비 스마트폰 비중: 24.26%


# 공통전처리1. 완전 중복 행 제거

In [6]:
# 5. 완전 중복 행 제거
df_11 = df0_11.drop_duplicates().copy()

In [7]:
print("중복 제거 전 행 수:", df0_11.shape[0])
print("중복 제거 후 행 수:", df_11.shape[0])
print("제거된 행 수:", df0_11.shape[0] - df_11.shape[0])
print("전체대비 삭제비율:", round((df0_11.shape[0] - df_11.shape[0]) / df0_11.shape[0] * 100, 2), "%")

중복 제거 전 행 수: 67501979
중복 제거 후 행 수: 67401460
제거된 행 수: 100519
전체대비 삭제비율: 0.15 %


# 공통전처리2. 동일고객이 동일세션에서 동일한 상품을 반복해서 구매할때

- 카테고리별 중복구매수 iqr 상한값을 구한 후
- 동일유저 + 동일상품 + 동일 세션에서 각 카테고리별 상한값 초과 반복 구매한 세션을 골라
- 짧은시간(0~90초) 안에 재 구매한 세션을 최종 이상후보 세션이라고 가정하였다

### 카테고리 공통 전처리후 실시해주세요 (현재는 전부 unknown 으로 처리)

In [8]:
# purchase 행만 추출
df_purchase_11 = df_11[df_11['event_type'] == 'purchase'].copy()

# category_code 결측치 처리
df_purchase_11['category_code'] = df_purchase_11['category_code'].fillna('unknown')

# 동일 유저 + 동일 상품 + 동일 세션 기준 구매 횟수 계산
purchase_sess_11 = (
    df_purchase_11
    .groupby(['category_code', 'user_id', 'product_id', 'user_session'])
    .size()
    .reset_index(name='buy_cnt')
)

# 2회 이상 반복 구매한 경우만 추출
purchase_sess_2_11 = purchase_sess_11[purchase_sess_11['buy_cnt'] >= 2].copy()

# 카테고리별 중복 구매수 IQR 상한값 계산
cat_iqr_11 = (
    purchase_sess_2_11
    .groupby('category_code')['buy_cnt']
    .agg(
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75))
    .reset_index())

cat_iqr_11['iqr'] = cat_iqr_11['q3'] - cat_iqr_11['q1']
cat_iqr_11['upper'] = cat_iqr_11['q3'] + 1.5 * cat_iqr_11['iqr']

# 방금 구한 purchase_sess_2에 카테고리별 upper 붙이기
purchase_sess_2_11 = purchase_sess_2_11.merge(
    cat_iqr_11[['category_code', 'upper']],
    on='category_code',
    how='left')

# 카테고리별 IQR 상한값 초과인 조합만 추출
purchase_sess_out_11 = purchase_sess_2_11[
    purchase_sess_2_11['buy_cnt'] > purchase_sess_2_11['upper']
].copy()

print("동일 유저 + 동일 상품 + 동일 세션 기준 2회 이상 반복 구매 조합 수:", len(purchase_sess_2_11))
print("카테고리별 IQR 상한값 초과 조합 수:", len(purchase_sess_out_11))
print("전체 2회 이상 조합 대비 비율:", round(len(purchase_sess_out_11) / len(purchase_sess_2_11) * 100, 2), "%")

display(purchase_sess_out_11.head())


동일 유저 + 동일 상품 + 동일 세션 기준 2회 이상 반복 구매 조합 수: 46435
카테고리별 IQR 상한값 초과 조합 수: 8214
전체 2회 이상 조합 대비 비율: 17.69 %


,category_code,user_id,product_id,user_session,buy_cnt,upper
32,accessories.bag,565476622,18300754,b9b693bd-db9d-435c-b72c-9aec7974fec3,3,2.00
36,accessories.bag,570218906,18300965,7e825517-f315-4044-b056-83a04bdef5b8,3,2.00
42,accessories.bag,572447972,46900046,b7667747-b8f7-4c5e-8981-883c69a386f4,3,2.00
44,accessories.bag,572716604,18300025,091b4cd6-9ddd-4bb7-b04a-1b0e826a85ee,3,2.00
61,accessories.wallet,563201420,28300609,c2cff788-ba39-41c4-bb66-f8bf15ca118d,3,2.00


In [9]:
# 카테고리별 최소/최대 iqr값
 
cat_min_max_11 = (
    purchase_sess_out_11
    .groupby('category_code')['buy_cnt']
    .agg(['min', 'max'])
    .reset_index()
)

display(cat_min_max_11.head(20))
print("이상 조합의 최소 구매 횟수:", purchase_sess_out_11['buy_cnt'].min())
print("이상 조합의 최대 구매 횟수:", purchase_sess_out_11['buy_cnt'].max())

# 카테고리별 IQR 상한값에서
# 몇 회부터 이상치가 되는지 계산
cat_cut_11 = cat_iqr_11.copy()
cat_cut_11['out_start'] = np.floor(cat_cut_11['upper']).astype(int) + 1

# 횟수별로 카테고리가 몇 개인지 보기
cat_cut_cnt_11 = (
    cat_cut_11
    .groupby('out_start')['category_code']
    .nunique()
    .reset_index(name='cat_cnt')
    .sort_values('out_start')
)

display(cat_cut_cnt_11)


,category_code,min,max
0,accessories.bag,3,3
1,accessories.wallet,3,3
2,apparel.shoes,3,14
3,apparel.shoes.keds,3,5
4,appliances.environment.air_conditioner,3,6
5,appliances.environment.air_heater,3,15
6,appliances.environment.vacuum,3,7
7,appliances.environment.water_heater,3,5
8,appliances.iron,3,7
9,appliances.ironing_board,3,5


이상 조합의 최소 구매 횟수: 3
이상 조합의 최대 구매 횟수: 54


,out_start,cat_cnt
0,3,100
1,4,5
2,5,5
3,6,1


한 세션안에서 같은 카테고리의 같은 상품을 반복하여 구매한 횟수에대해 iqr 상한값을 적용 했을 때
100개의 카테고리는 3회구매부터 이상치로 잡혔다.

In [10]:
# 카테고리별 이상치 시작 횟수 계산
cat_cut_11 = cat_iqr_11.copy()
cat_cut_11['out_start'] = np.floor(cat_cut_11['upper']).astype(int) + 1

# 4, 5, 6회부터 이상치로 잡히는 카테고리만 보기
cat_cut_456_11 = cat_cut_11[cat_cut_11['out_start'].isin([4, 5, 6])].copy()

cat_cut_456_11 = cat_cut_456_11.sort_values(['out_start', 'category_code'])

display(cat_cut_456_11[['category_code', 'upper', 'out_start']])


,category_code,upper,out_start
5,apparel.glove,3.00,4
29,appliances.kitchen.juicer,3.25,4
40,appliances.personal.massager,3.88,4
50,auto.accessories.winch,3.50,4
84,electronics.video.projector,3.38,4
14,apparel.tshirt,4.00,5
52,computers.components.cpu,4.50,5
87,furniture.bathroom.toilet,4.50,5
101,kids.swing,4.50,5
106,sport.snowboard,4.50,5


이상치가 4,5,6 개로 잡힌 항목은 위와같으며, 스마트폰의 이상후보 조합은 아래와 같았다

In [11]:
purchase_sess_out_11[purchase_sess_out_11['category_code'] == 'electronics.smartphone'] \
    .sort_values('buy_cnt', ascending=False)

,category_code,user_id,product_id,user_session,buy_cnt,upper
14376,electronics.smartphone,518514099,1005116,1d34878d-1a42-401b-90a4-d44e2aa1e127,54,2.00
25805,electronics.smartphone,564068124,1004767,3b00665a-daff-4a2c-bba2-a152cc6e62c9,32,2.00
11315,electronics.smartphone,513190540,1004848,7ece423b-f354-4c1d-ace8-ba0372ab96e2,24,2.00
25843,electronics.smartphone,564068124,1004833,384088f7-1f2f-4179-af62-7958bbd171b1,23,2.00
21475,electronics.smartphone,549498325,1004856,f976730c-33df-4f53-b920-9da78eec2ac4,17,2.00
...,...,...,...,...,...,...
34013,electronics.smartphone,579589235,1004838,de0bc5ef-545d-44a0-98ea-c0b34180234b,3,2.00
10437,electronics.smartphone,512396132,1005124,3151e95b-0243-4422-bae5-7628f85c4ca6,3,2.00
10433,electronics.smartphone,512393374,1005100,ba4e6c4a-8d90-467a-b45b-548d82908ec4,3,2.00
10419,electronics.smartphone,512388750,1005107,c5fc90a2-3797-4c83-a9c7-0eec32b9a040,3,2.00


#### 스마트폰 카테고리는 IQR 상한값이 2.0으로 계산되어, 3회 구매부터 1차 이상 후보로 분류되었지만 실제 최대 반복 구매 횟수는 54회로 나타나 상한값과의 격차가 11월에도 매우 컸다.

In [12]:
# 카테고리별로 실제 최대 구매횟수와 IQR 상한값 차이 계산
cat_gap_11 = (
    purchase_sess_out_11
    .groupby('category_code')['buy_cnt']
    .max()
    .reset_index(name='max_buy_cnt')
    .merge(cat_iqr_11[['category_code', 'upper']], on='category_code', how='left')
)

cat_gap_11['over_gap'] = cat_gap_11['max_buy_cnt'] - cat_gap_11['upper']

# 상한값보다 과도하게 큰 카테고리 top 20
cat_gap_top20_11 = cat_gap_11.sort_values('over_gap', ascending=False).head(20)

display(cat_gap_top20_11)


,category_code,max_buy_cnt,upper,over_gap
64,electronics.smartphone,54,2.00,52.00
57,electronics.audio.headphone,23,2.00,21.00
83,unknown,23,2.00,21.00
22,appliances.kitchen.refrigerators,16,2.00,14.00
5,appliances.environment.air_heater,15,2.00,13.00
2,apparel.shoes,14,2.00,12.00
67,electronics.video.tv,14,2.00,12.00
46,computers.peripherals.monitor,13,2.00,11.00
44,computers.notebook,11,2.00,9.00
41,computers.components.videocards,11,2.00,9.00


#### iqr 상한값보다 과도하게 큰 카테고리 top 20위 10월과는 다른 항목이지만 스마트폰이 가장 눈에 띄었다

### 해당 상품들을 전부 이상치라고 볼수 없기때문에 두가지 조건을 줬다
### 1. 뷰, 카트 없이 "구매" 만 반복된 행 확인


In [13]:
# purchase_sess_out_11 기준 조합의 전체 로그만 보기
out_log_11 = df_11.merge(
    purchase_sess_out_11[['category_code', 'user_id', 'product_id', 'user_session']].drop_duplicates(),
    on=['category_code', 'user_id', 'product_id', 'user_session'],
    how='inner'
)

# view, cart 없이 purchase만 있는 조합 찾기
out_check_11 = (
    out_log_11
    .groupby(['category_code', 'user_id', 'product_id', 'user_session', 'event_type'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['purchase']:
    if col not in out_check_11.columns:
        out_check_11[col] = 0

out_purchase_only_11 = out_check_11[
    (out_check_11['view'] == 0) &
    (out_check_11['cart'] == 0) &
    (out_check_11['purchase'] >= 2)
].copy()

print("purchase_sess_out_11 조합 수:", len(purchase_sess_out_11))
print("view/cart 없이 purchase만 반복된 조합 수:", len(out_purchase_only_11))

display(out_purchase_only_11.head(20))


purchase_sess_out_11 조합 수: 8214
view/cart 없이 purchase만 반복된 조합 수: 2


event_type,category_code,user_id,product_id,user_session,cart,purchase,view
1745,electronics.clocks,567059021,5100337,32d47174-849e-46ee-91b3-16be02c607dc,0,3,0
4150,electronics.smartphone,554571511,1005161,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac,0,5,0


In [14]:
# 2건 확인
 
target_sessions = out_purchase_only_11['user_session'].drop_duplicates()

df_11[
    df_11['user_session'].isin(target_sessions)
].sort_values(['user_session', 'event_time'])

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
44380044,2019-11-17 16:01:55 UTC,view,1005160,2053013555631882655,electronics.smartphone,xiaomi,202.06,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
44388735,2019-11-17 16:03:08 UTC,view,1005160,2053013555631882655,electronics.smartphone,xiaomi,202.06,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
44420183,2019-11-17 16:07:31 UTC,purchase,1005161,2053013555631882655,electronics.smartphone,xiaomi,202.32,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
44429006,2019-11-17 16:08:44 UTC,purchase,1005161,2053013555631882655,electronics.smartphone,xiaomi,202.32,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
44437952,2019-11-17 16:09:58 UTC,purchase,1005161,2053013555631882655,electronics.smartphone,xiaomi,202.32,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
44523206,2019-11-17 16:21:22 UTC,purchase,1005161,2053013555631882655,electronics.smartphone,xiaomi,202.32,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
44539632,2019-11-17 16:23:35 UTC,purchase,1005161,2053013555631882655,electronics.smartphone,xiaomi,202.32,554571511,04ff6ec3-e8aa-2588-1ddb-e11f8cc886ac
40273382,2019-11-17 06:03:50 UTC,purchase,5100337,2053013553341792533,electronics.clocks,apple,421.38,567059021,32d47174-849e-46ee-91b3-16be02c607dc
40276769,2019-11-17 06:04:49 UTC,purchase,5100337,2053013553341792533,electronics.clocks,apple,421.38,567059021,32d47174-849e-46ee-91b3-16be02c607dc
40279428,2019-11-17 06:05:24 UTC,purchase,5100337,2053013553341792533,electronics.clocks,apple,421.38,567059021,32d47174-849e-46ee-91b3-16be02c607dc


#### 결론 : 반복 구매가 카테고리 iqr 상한값보다 크면서 purchase 만 반복된 조합은 비정상 후보로 판단하여 모든 분석에서 제외하고 view 또는 cart 가 하나라도 존재하는 조합은 이상치 후보 플래그로 처리한다

### 2. 같은 유저 / 같은 세션 / 같은 상품 내에서 purchase 사이의 시간 간격을 구간별로 나눠서 확인하여 이상치 후보를 결정하였다

In [15]:
# purchase_sess_out_11 에 해당하는 purchase 행만 다시 가져오기
outlier_buy_11 = df_purchase_11.merge(
    purchase_sess_out_11[['category_code', 'user_id', 'product_id', 'user_session']],
    on=['category_code', 'user_id', 'product_id', 'user_session'],
    how='inner'
).copy()

# 시간형식 변환
outlier_buy_11['event_time'] = pd.to_datetime(outlier_buy_11['event_time'])

# 같은 유저 + 상품 + 세션 기준 시간순 정렬
outlier_buy_11 = outlier_buy_11.sort_values(
    ['user_id', 'user_session', 'product_id', 'event_time']
).copy()

# 바로 이전 purchase 시점 구하기
outlier_buy_11['prev_time'] = outlier_buy_11.groupby(
    ['user_id', 'user_session', 'product_id']
)['event_time'].shift(1)

# 재구매 간격(초) 계산
outlier_buy_11['gap_sec'] = (
    outlier_buy_11['event_time'] - outlier_buy_11['prev_time']
).dt.total_seconds()

# 실제 재구매 간격이 있는 purchase만 보기
gap_log_11 = outlier_buy_11[outlier_buy_11['gap_sec'].notna()].copy()

# 구간별 분포표 만들기
gap_log_11['gap_range'] = pd.cut(
    gap_log_11['gap_sec'],
    bins=[0, 30, 60, 90, 180, 300, 600, 1800, 3600, gap_log_11['gap_sec'].max() + 1],
    right=False,
    include_lowest=True
)

gap_dist_11 = (
    gap_log_11['gap_range']
    .value_counts()
    .sort_index()
    .reset_index()
)
gap_dist_11.columns = ['gap_range', 'cnt']
gap_dist_11['ratio(%)'] = (gap_dist_11['cnt'] / gap_dist_11['cnt'].sum() * 100).round(2)

display(gap_dist_11)

,gap_range,cnt,ratio(%)
0,"[0.0, 30.0)",165,0.79
1,"[30.0, 60.0)",4104,19.75
2,"[60.0, 90.0)",6003,28.88
3,"[90.0, 180.0)",5868,28.23
4,"[180.0, 300.0)",2150,10.34
5,"[300.0, 600.0)",1388,6.68
6,"[600.0, 1800.0)",728,3.50
7,"[1800.0, 3600.0)",173,0.83
8,"[3600.0, 2345642.0)",205,0.99


#### 60초 이후 구간도 확인할 필요가 있다고 판단하여, 재구매 이상치 후보를 0~90초까지 확장해서 보고 해당 구간을 10초 간격으로 세분화해 표를 확인했다

In [16]:
gap_log_11 = df_purchase_11.merge(
    purchase_sess_out_11[['category_code', 'user_id', 'product_id', 'user_session']].drop_duplicates(),
    on=['category_code', 'user_id', 'product_id', 'user_session'],
    how='inner'
).copy()

gap_log_11['event_time'] = pd.to_datetime(gap_log_11['event_time'])

gap_log_11 = gap_log_11.sort_values(
    ['category_code', 'user_id', 'product_id', 'user_session', 'event_time']
).copy()

gap_log_11['prev_time'] = gap_log_11.groupby(
    ['category_code', 'user_id', 'product_id', 'user_session']
)['event_time'].shift(1)

gap_log_11['gap_sec'] = (
    gap_log_11['event_time'] - gap_log_11['prev_time']
).dt.total_seconds()

gap_log_11 = gap_log_11[
    gap_log_11['gap_sec'].notna() &
    (gap_log_11['gap_sec'] >= 0) &
    (gap_log_11['gap_sec'] < 90)
].copy()

gap_log_11['gap_range'] = pd.cut(
    gap_log_11['gap_sec'],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90],
    right=False,
    include_lowest=True
)

gap_dist_11 = (
    gap_log_11['gap_range']
    .value_counts()
    .sort_index()
    .reset_index()
)
gap_dist_11.columns = ['gap_range', 'cnt']
gap_dist_11['ratio(%)'] = (gap_dist_11['cnt'] / gap_dist_11['cnt'].sum() * 100).round(2)

display(gap_dist_11)

,gap_range,cnt,ratio(%)
0,"[0, 10)",16,0.16
1,"[10, 20)",41,0.40
2,"[20, 30)",108,1.05
3,"[30, 40)",465,4.53
4,"[40, 50)",1421,13.83
5,"[50, 60)",2218,21.59
6,"[60, 70)",2336,22.74
7,"[70, 80)",2027,19.73
8,"[80, 90)",1640,15.97


#### 0~30초구간 상세

In [17]:
cat_gap_top20_11['all_cnt'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_11['drop_30_cnt'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 30),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_11['keep_30_cnt'] = cat_gap_top20_11['all_cnt'] - cat_gap_top20_11['drop_30_cnt']

cat_gap_top20_11['max_after_30'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 30),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_11[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_30', 'all_cnt', 'drop_30_cnt', 'keep_30_cnt']
    ]
)



,category_code,upper,max_buy_cnt,max_after_30,all_cnt,drop_30_cnt,keep_30_cnt
64,electronics.smartphone,2.00,54,54,4509,69,4440
57,electronics.audio.headphone,2.00,23,23,548,14,534
83,unknown,2.00,23,23,1511,24,1487
22,appliances.kitchen.refrigerators,2.00,16,16,99,2,97
5,appliances.environment.air_heater,2.00,15,15,13,0,13
2,apparel.shoes,2.00,14,14,64,3,61
67,electronics.video.tv,2.00,14,14,259,9,250
46,computers.peripherals.monitor,2.00,13,13,18,1,17
44,computers.notebook,2.00,11,11,187,3,184
41,computers.components.videocards,2.00,11,11,11,1,10


### 표 해석방법
### electronics.clocks 기준 0~30초 조합을 삭제해도 가장 많이구매한 54회가 유지되었고
### 전체 조합중 제거되는조합은 69개뿐이였다
### 각 구간별로 확인해봤다 

#### 0~40초구간

In [18]:
cat_gap_top20_11['all_cnt'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_11['drop_40_cnt'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 40),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_11['keep_40_cnt'] = cat_gap_top20_11['all_cnt'] - cat_gap_top20_11['drop_40_cnt']

cat_gap_top20_11['max_after_40'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 40),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_11[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_40', 'all_cnt', 'drop_40_cnt', 'keep_40_cnt']
    ]
)

,category_code,upper,max_buy_cnt,max_after_40,all_cnt,drop_40_cnt,keep_40_cnt
64,electronics.smartphone,2.00,54,54,4509,244,4265
57,electronics.audio.headphone,2.00,23,12,548,54,494
83,unknown,2.00,23,23,1511,117,1394
22,appliances.kitchen.refrigerators,2.00,16,16,99,7,92
5,appliances.environment.air_heater,2.00,15,15,13,0,13
2,apparel.shoes,2.00,14,5,64,6,58
67,electronics.video.tv,2.00,14,14,259,16,243
46,computers.peripherals.monitor,2.00,13,13,18,4,14
44,computers.notebook,2.00,11,11,187,16,171
41,computers.components.videocards,2.00,11,11,11,2,9


#### 0~50초구간

In [19]:
cat_gap_top20_11['drop_50_cnt'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 50),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_11['keep_50_cnt'] = cat_gap_top20_11['all_cnt'] - cat_gap_top20_11['drop_50_cnt']

cat_gap_top20_11['max_after_50'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 50),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_11[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_50', 'all_cnt', 'drop_50_cnt', 'keep_50_cnt']
    ]
)


,category_code,upper,max_buy_cnt,max_after_50,all_cnt,drop_50_cnt,keep_50_cnt
64,electronics.smartphone,2.00,54,17,4509,748,3761
57,electronics.audio.headphone,2.00,23,12,548,151,397
83,unknown,2.00,23,15,1511,312,1199
22,appliances.kitchen.refrigerators,2.00,16,16,99,16,83
5,appliances.environment.air_heater,2.00,15,4,13,2,11
2,apparel.shoes,2.00,14,5,64,15,49
67,electronics.video.tv,2.00,14,14,259,41,218
46,computers.peripherals.monitor,2.00,13,13,18,10,8
44,computers.notebook,2.00,11,6,187,49,138
41,computers.components.videocards,2.00,11,6,11,4,7


#### 0~60초구간

In [20]:
cat_gap_top20_11['drop_60_cnt'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 60),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='inner'
    )
    .groupby('category_code')
    .size()
).fillna(0).astype(int)

cat_gap_top20_11['keep_60_cnt'] = cat_gap_top20_11['all_cnt'] - cat_gap_top20_11['drop_60_cnt']

cat_gap_top20_11['max_after_60'] = cat_gap_top20_11['category_code'].map(
    purchase_sess_out_11
    .merge(
        gap_log_11.loc[
            (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 60),
            ['category_code', 'user_id', 'product_id', 'user_session']
        ].drop_duplicates(),
        on=['category_code', 'user_id', 'product_id', 'user_session'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .groupby('category_code')['buy_cnt']
    .max()
).fillna(0).astype(int)

display(
    cat_gap_top20_11[
        ['category_code', 'upper', 'max_buy_cnt', 'max_after_60', 'all_cnt', 'drop_60_cnt', 'keep_60_cnt']
    ]
)

,category_code,upper,max_buy_cnt,max_after_60,all_cnt,drop_60_cnt,keep_60_cnt
64,electronics.smartphone,2.00,54,14,4509,1404,3105
57,electronics.audio.headphone,2.00,23,11,548,256,292
83,unknown,2.00,23,9,1511,537,974
22,appliances.kitchen.refrigerators,2.00,16,16,99,23,76
5,appliances.environment.air_heater,2.00,15,4,13,2,11
2,apparel.shoes,2.00,14,5,64,29,35
67,electronics.video.tv,2.00,14,9,259,68,191
46,computers.peripherals.monitor,2.00,13,3,18,14,4
44,computers.notebook,2.00,11,6,187,81,106
41,computers.components.videocards,2.00,11,5,11,6,5


#### 재구매 간격분포를 60초까지 확인해본결과 40초 이후 구간부터 cnt 갯수가 증가하는 흐름이 나타났고 50초 구간부터 반복구매 감소가 눈에 띄게 나타났다 (스마트폰 반복구매횟수 54 -> 17), 이 범위를 60초구간까지 잡으면 반복구매 최대값은 17회에서 14회로 소폭감소하지만 삭제되는 조합의수는 40~50초 1421, 50~60초 2218 으로 1.6배 증가하여 

#### 스마트폰 반복구매횟수 14~17회인 로그조합이 50~60초중 어디에 분포되어있는지 확인해봤다

In [21]:
pair_cols = ['category_code', 'user_id', 'product_id', 'user_session']

# -------------------------------------------------
# 1) 50초 컷 삭제 후보
#    0초 이상 ~ 50초 미만 gap이 있는 조합
# -------------------------------------------------
drop_pairs_50_11 = gap_log_11.loc[
    (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 50),
    pair_cols
].drop_duplicates()

# -------------------------------------------------
# 2) 50초 컷 이후 남은 조합
# -------------------------------------------------
keep_50_11 = (
    purchase_sess_out_11
    .merge(drop_pairs_50_11, on=pair_cols, how='left', indicator=True)
    .query("_merge == 'left_only'")
    .drop(columns='_merge')
    .copy()
)

# -------------------------------------------------
# 3) 50~60초 gap이 실제로 존재하는 조합
# -------------------------------------------------
gap_pairs_50_60_11 = gap_log_11.loc[
    (gap_log_11['gap_sec'] >= 50) &
    (gap_log_11['gap_sec'] < 60),
    pair_cols
].drop_duplicates()

# -------------------------------------------------
# 4) 50초 컷 이후 남은 조합 중
#    스마트폰 + buy_cnt 14~17 + 50~60초 gap 존재 조합만 추출
# -------------------------------------------------
smart_14_17_50_60_cases_11 = (
    keep_50_11[
        (keep_50_11['category_code'] == 'electronics.smartphone') &
        (keep_50_11['buy_cnt'].between(14, 17))
    ]
    .merge(
        gap_pairs_50_60_11,
        on=pair_cols,
        how='inner'
    )
    .copy()
)

smart_14_17_50_60_cases_11 = smart_14_17_50_60_cases_11.sort_values(
    'buy_cnt',
    ascending=False
)

print(
    "스마트폰 14~17회 반복구매 조합 수(50~60초간격) :",
    len(smart_14_17_50_60_cases_11)
)

display(smart_14_17_50_60_cases_11)

스마트폰 14~17회 반복구매 조합 수(50~60초간격) : 4


,category_code,user_id,product_id,user_session,buy_cnt,upper
3,electronics.smartphone,570500831,1005160,b76997bb-be5c-458d-a8cf-65754d9d9986,17,2.00
0,electronics.smartphone,518514099,1005116,bb3b8b49-6698-43d1-8b4b-d2928d474eff,15,2.00
2,electronics.smartphone,564068124,1004833,339cc71d-85fd-42c4-a758-e0b38ce2f3cc,15,2.00
1,electronics.smartphone,562627150,1004767,1228f697-ef7f-44e9-a443-e792d836efd8,14,2.00


In [22]:
pair_cols = ['category_code', 'user_id', 'product_id', 'user_session']

# -------------------------------------------------
# 1) 스마트폰 14~17회 반복구매 조합 기준으로 전체 gap 가져오기
#    기준 데이터: smart_14_17_50_60_cases_11
# -------------------------------------------------
smart_14_17_all_gap_detail_11 = (
    gap_log_11
    .merge(
        smart_14_17_50_60_cases_11[pair_cols + ['buy_cnt']],
        on=pair_cols,
        how='inner'
    )
    .copy()
)

# 초 단위 정수 변환
smart_14_17_all_gap_detail_11['gap_sec_int'] = (
    smart_14_17_all_gap_detail_11['gap_sec']
    .astype(int)
)

# -------------------------------------------------
# 2) 조합별 전체 gap 리스트 만들기
# -------------------------------------------------
combo_all_gap_where_11 = (
    smart_14_17_all_gap_detail_11
    .groupby(pair_cols + ['buy_cnt'])['gap_sec_int']
    .apply(lambda x: sorted(x.tolist()))
    .reset_index(name='all_gap_seconds')
)

combo_all_gap_where_11['all_gap_cnt'] = (
    combo_all_gap_where_11['all_gap_seconds']
    .apply(len)
)

combo_all_gap_where_11 = combo_all_gap_where_11.sort_values(
    ['buy_cnt', 'all_gap_cnt'],
    ascending=[False, False]
)

print(
    "스마트폰 14~17회 반복구매 조합별 전체 gap 현황:",
    len(combo_all_gap_where_11)
)

display(combo_all_gap_where_11)

스마트폰 14~17회 반복구매 조합별 전체 gap 현황: 4


,category_code,user_id,product_id,user_session,buy_cnt,all_gap_seconds,all_gap_cnt
3,electronics.smartphone,570500831,1005160,b76997bb-be5c-458d-a8cf-65754d9d9986,17,"[51, 51, 56, 61, 63, 63, 66, 68, 69, 75, 76, 7...",14
0,electronics.smartphone,518514099,1005116,bb3b8b49-6698-43d1-8b4b-d2928d474eff,15,"[56, 60, 65, 68, 71, 72, 74, 80, 80, 85, 86]",11
2,electronics.smartphone,564068124,1004833,339cc71d-85fd-42c4-a758-e0b38ce2f3cc,15,"[53, 53, 54, 59, 63, 67, 69, 70, 87]",9
1,electronics.smartphone,562627150,1004767,1228f697-ef7f-44e9-a443-e792d836efd8,14,"[51, 53, 56, 57, 59, 59, 62, 68, 73, 74]",10


17회 반복구매 조합에서는 51~56초 간격이 3회 확인되었으며, 다른 세션의 구매간격도 51~59초까지로 들쭉날쭉하였다

50~60초 사이의 반복구매 조합을 확인했다.

In [23]:
# -------------------------------------------------
# 50초 이상 ~ 60초 미만 구간만 1초 단위로 확인
# 기준 데이터: gap_log_11
# -------------------------------------------------

gap_50_60_11 = gap_log_11[
    (gap_log_11['gap_sec'] >= 50) &
    (gap_log_11['gap_sec'] < 60)
].copy()

# 1초 단위 구간 생성: [50,51), [51,52) ... [59,60)
gap_50_60_11['gap_1sec_range'] = pd.cut(
    gap_50_60_11['gap_sec'],
    bins=list(range(50, 61)),
    right=False,
    include_lowest=True
)

gap_50_60_dist_11 = (
    gap_50_60_11['gap_1sec_range']
    .value_counts()
    .sort_index()
    .reset_index()
)

gap_50_60_dist_11.columns = ['gap_1sec_range', 'cnt']

gap_50_60_dist_11['ratio(%)'] = (
    gap_50_60_dist_11['cnt'] / gap_50_60_dist_11['cnt'].sum() * 100
).round(2)

display(gap_50_60_dist_11)

,gap_1sec_range,cnt,ratio(%)
0,"[50, 51)",180,8.12
1,"[51, 52)",202,9.11
2,"[52, 53)",219,9.87
3,"[53, 54)",179,8.07
4,"[54, 55)",263,11.86
5,"[55, 56)",243,10.96
6,"[56, 57)",215,9.69
7,"[57, 58)",247,11.14
8,"[58, 59)",218,9.83
9,"[59, 60)",252,11.36


### 50~60초 구간을 1초 단위로 확인한 결과, 특정 초 구간에 집중되기보다는 전체적으로 8~12% 수준으로 고르게 분포하였다. 따라서 55초와 같이 중간 기준을 추가로 설정할 명확한 근거는 부족하였다. 

### 반복구매 최대값이 크게 감소하는 50초 까지를 경계구간으로 해석하였고 이에 따라 플래그 컬럼을 만들어 후 분석목적에 맞게 적용하려 한다

In [24]:
pair_cols = ['category_code', 'user_id', 'product_id', 'user_session']

# ── Step 1: purchase만 반복된 세션 분류 (out_purchase_only_11 활용) ──
problem_sessions = out_purchase_only_11['user_session'].drop_duplicates()

session_view_check = (
    df_11[df_11['user_session'].isin(problem_sessions)]
    .groupby('user_session')['event_type']
    .apply(lambda x: (x == 'view').any())
    .reset_index(name='has_view')
)

del_sessions  = session_view_check[~session_view_check['has_view']]['user_session']  # 32d47174 (3행)
keep_sessions = session_view_check[ session_view_check['has_view']]['user_session']  # 04ff6ec3 (7행)

df_11 = df_11[~df_11['user_session'].isin(del_sessions)].copy()

# ── Step 2: 플래그 초기화 ────────────────────────────────────
df_11 = df_11.drop(columns=['fsec_11'], errors='ignore')
df_11['event_time']      = pd.to_datetime(df_11['event_time'])
gap_log_11['prev_time']  = pd.to_datetime(gap_log_11['prev_time'])
gap_log_11['event_time'] = pd.to_datetime(gap_log_11['event_time'])
gap_log_11 = gap_log_11[~gap_log_11['user_session'].isin(del_sessions)].copy()

# ── Step 3: gap_log 기반 [prev_time, cur_time] 구간 플래그 ──
df_11 = df_11.reset_index(drop=True)
df_11['_idx'] = df_11.index

t_0_40_11 = gap_log_11.loc[
    (gap_log_11['gap_sec'] >= 0) & (gap_log_11['gap_sec'] < 40),
    pair_cols + ['prev_time', 'event_time']
].rename(columns={'event_time': 'cur_time'})

t_40_50_11 = gap_log_11.loc[
    (gap_log_11['gap_sec'] >= 40) & (gap_log_11['gap_sec'] < 50),
    pair_cols + ['prev_time', 'event_time']
].rename(columns={'event_time': 'cur_time'})

idx_0_40 = (
    df_11.merge(t_0_40_11, on=pair_cols, how='inner')
    .query('event_time > prev_time and event_time <= cur_time')
    ['_idx'].unique()
)
idx_40_50 = (
    df_11.merge(t_40_50_11, on=pair_cols, how='inner')
    .query('event_time >= prev_time and event_time <= cur_time')
    ['_idx'].unique()
)

df_11['fsec_11'] = None
df_11.loc[idx_0_40,  'fsec_11'] = '0_40'
df_11.loc[idx_40_50, 'fsec_11'] = '40_50'

# ── Step 4: keep 세션 플래그 (마지막에 덮어씌움) ─────────────
df_11.loc[df_11['user_session'].isin(keep_sessions), 'fsec_11'] = 'keep'

df_11 = df_11.drop(columns=['_idx'])

print(df_11['fsec_11'].value_counts(dropna=False))

# ── Step 5: 구간별 요약 테이블 ───────────────────────────────
total_view     = (df_11['event_type'] == 'view').sum()
total_cart     = (df_11['event_type'] == 'cart').sum()
total_purchase = (df_11['event_type'] == 'purchase').sum()

def make_summary(label, lo, hi, custom_log=None):
    if custom_log is not None:
        log   = custom_log
        pairs = log[pair_cols].drop_duplicates()
    else:
        pairs = gap_log_11.loc[
            (gap_log_11['gap_sec'] >= lo) & (gap_log_11['gap_sec'] < hi),
            pair_cols
        ].drop_duplicates()
        log = df_11.merge(pairs, on=pair_cols, how='inner')

    v = (log['event_type'] == 'view').sum()
    c = (log['event_type'] == 'cart').sum()
    p = (log['event_type'] == 'purchase').sum()

    return {
        '구간': label,
        '조합수': len(pairs),
        '전체행수': len(log),
        'view수': v,
        'cart수': c,
        'purchase수': p,
        '전체 view 중 비율(%)':     round(v / total_view     * 100, 6) if total_view     > 0 else 0,
        '전체 cart 중 비율(%)':     round(c / total_cart     * 100, 6) if total_cart     > 0 else 0,
        '전체 purchase 중 비율(%)': round(p / total_purchase * 100, 6) if total_purchase > 0 else 0,
    }

log_keep_11 = df_11[df_11['user_session'].isin(keep_sessions)].copy()

summary_df_11 = pd.DataFrame([
    make_summary('0~40초',  0,  40),
    make_summary('40~50초', 40, 50),
    make_summary('keep',    None, None, custom_log=log_keep_11),
])

display(summary_df_11)

fsec_11
None     67396022
40_50        4225
0_40         1203
keep            7
Name: count, dtype: int64


,구간,조합수,전체행수,view수,cart수,purchase수,전체 view 중 비율(%),전체 cart 중 비율(%),전체 purchase 중 비율(%)
0,0~40초,543,5627,2403,1522,1702,0.00,0.05,0.19
1,40~50초,1147,13119,5516,3739,3864,0.01,0.13,0.42
2,keep,2,7,2,0,5,0.00,0.00,0.00
